# Raciocício Probabilístico

### Disciplina de Sistemas Inteligentes

### Departamento de Informática e Estatística

### Profa. Nathalia da Cruz Alves


**Nota 1**: Em caso de reprodução, publicação ou compartilhamento deste material, a autoria deverá ser obrigatoriamente mantida.

**Nota 2**: Este material foi elaborado com o auxílio de ferramentas de Inteligência Artificial (IA), utilizadas para o refinamento de explicações e revisão de trechos de código. A autoria e a responsabilidade pelo conteúdo final permanecem com a autora.

Estudante(s): insira seu nome aqui


**Observação**: Solicita-se que sejam informados apenas os nomes dos estudantes efetivamente participantes da atividade.

### Classificação de mensagens com Naive Bayes


Ao final da atividade, você deverá ser capaz de:
- compreender o problema de classificação de textos
- preparar dados textuais para um modelo de Machine Learning
- transformar textos em representações numéricas com Bag of Words
- entender a ideia do Teorema de Bayes
- treinar um classificador MultinomialNB
- avaliar o modelo usando accuracy, precision, recall e F1-score
- interpretar os erros cometidos pelo classificador usando uma matriz de confusão



### Configurações iniciais

In [1]:
import pandas as pd
import zipfile
import requests
from io import BytesIO

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

## Contextualização

Mensagens indesejadas, como propagandas enganosas e tentativas de fraude, podem ser identificadas automaticamente por sistemas de classificação de texto.

Nesta atividade, você vai construir um modelo capaz de classificar mensagens em duas categorias: spam e não spam.

## Conjunto de dados

O conjunto de dados a ser utilizado está disponível no repositório da University of California, que conta com uma coleção de conjuntos de dados para fins de pesquisa experimental.

Link:
https://archive.ics.uci.edu/dataset/228/sms+spam+collection


In [2]:
# @title Download dos dados
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

response = requests.get(url)
print(response.status_code)

with open("sms_spam_collection.zip", "wb") as f:
    f.write(response.content)


with zipfile.ZipFile("sms_spam_collection.zip", "r") as zip_ref:
    zip_ref.extractall("sms_spam_collection")

200


In [3]:
# @title Carregamento dos dados

df = pd.read_csv(
    "sms_spam_collection/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["classe", "mensagem"]
)

## Análise Exploratória de Dados (Exploratory Data Analysis)

Antes de construir o modelo, é importante explorar o conjunto de dados.

A Análise Exploratória de Dados (EDA) ajuda a entender a estrutura dos dados, identificar possíveis problemas e observar características que podem ser relevantes para o problema de classificação.


Nesta etapa, será investigado:
- quantas mensagens existem no conjunto de dados
- como as mensagens estão distribuídas entre spam e ham
- se existem valores ausentes
- o tamanho das mensagens
- algumas características das mensagens de cada classe

Acesse a documentação da biblioteca pandas e utilize-a como base para construir códigos que respondam às perguntas a seguir.
https://pandas.pydata.org/docs/reference/frame.html

### Exercício 1. Dimensões do conjunto de dados
Quantas mensagens existem no conjunto de dados? Quantas colunas?



### Exercício 2. Conhecendo as colunas

Observe as primeiras linhas e verifique o tipo de dados.

- Qual coluna representa a classificação?

- Qual coluna contém o texto da mensagem?

- Os tipos de dados estão adequados?

### Exercício 3. Distribuição das classes

Verifique quantas mensagens pertencem a cada categoria.

Utilize o comando:

```
df["classe"].value_counts()
```


E para visualizar melhor

```
df["classe"].value_counts().plot(
    kind="bar",
    title="Distribuição das mensagens"
)
```

- Existem mais mensagens ham ou spam?

- As duas classes possuem quantidades semelhantes?

- O que essa diferença pode representar para a avaliação do modelo?



### Exercício 4. Valores ausentes

Para verificar se existem informações faltantes:


```
df.isnull().sum()
```

- Existem valores ausentes? Caso existam, em quais colunas?




### Exercício 5. Tamanho das mensagens

O texto também pode ser analisado numericamente. Crie uma nova coluna contendo a quantidade de caracteres de cada mensagem:


```
df["tamanho"] = df["mensagem"].str.len()
```

E observe algumas estatísticas:


```
df["tamanho"].describe()
```

Responda:

- Qual é o tamanho médio das mensagens?

- Qual é a menor mensagem?

- Qual é a maior mensagem?

- Você percebe alguma diferença entre mensagens spam e ham?






## Preparação dos rótulos

O algoritmo trabalhará com valores numéricos. Portanto, as classes serão representadas da seguinte forma:
- ham → 0
- spam → 1



In [ ]:
df["classe"] = df["classe"].map({
    "ham": 0,
    "spam": 1
})

## Separação entre treino e teste

Antes de treinar o modelo, é necessário separar os dados:
- Dados de treinamento: usados pelo modelo para aprender os padrões.
- Dados de teste: usados depois para verificar se o modelo consegue fazer previsões em dados que não utilizou durante o treinamento.

Essa separação é fundamental para avaliar se o modelo realmente aprendeu padrões ou simplesmente "decorou" os exemplos.

OBS: na prática, utilizamos outras abordagens de validação cruzada, em vez de apenas dividir em treino e teste. Porém, para o objetivo desta aula, dividiremos o conjunto em duas partes apenas.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["mensagem"],             # X
    df["classe"],               # y

    test_size=0.2,              # 20% dos dados serão separados para teste

    random_state=42,            # controla a aleatoriedade utilizada na divisão

    stratify=df["classe"]       # faz com que a proporção entre ham e spam seja
                                # aproximadamente preservada tanto no conjunto
                                # de treinamento quanto no conjunto de teste
)

## Bag of Words

O computador não consegue trabalhar diretamente com uma frase como nós. Para um algoritmo de Machine Learning, é necessário transformar o texto em números.

O Bag of Words é uma técnica simples para fazer isso. A ideia é:
- reunir todas as palavras que aparecem nos textos
- criar um vocabulário com essas palavras
- contar quantas vezes cada palavra aparece em cada texto
- representar cada texto por esses números

O nome Bag of Words significa  "saco de palavras". Imagine que colocamos todas as palavras de um texto dentro de um saco e contamos quantas vezes cada uma aparece. Nesse processo, a ordem das palavras não é considerada.


Sem utilizar nenhuma biblioteca, quais palavras aparecem nas frases a seguir? Quantas vezes cada palavra aparece?

In [ ]:
frases = [
    "gosto de estudar machine learning",
    "gosto de estudar Python",
    "Python é interessante"
]

O vocabulário seria:

- de
- estudar
- gosto
- interessante
- learning
- machine
- python
- é

As ocorrências em cada frase são contadas e isso pode pode ser representado por uma tabela.

Essa tabela é chamada de matriz documento-termo (document-term matrix).



In [ ]:
# Observando como o Bag of Words funciona
vectorizer = CountVectorizer()

matriz = vectorizer.fit_transform(frases)

print(vectorizer.get_feature_names_out())
print(matriz.toarray())

### Aplicando o BoW ao conjunto de dados

In [ ]:
vectorizer = CountVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

## Treinamento do Naive Bayes

Agora que as mensagens foram transoformadas em números usando Bag of Words, é possível treinar o classificador.

Treinar um modelo de Machine Learning significa apresentar exemplos ao algoritmo para que ele encontre padrões nos dados.

Neste problema, já sabemos a classificação das mensagens do conjunto de treinamento:

- 0: mensagem não spam (ham)

- 1: mensagem spam

O modelo recebe, portanto, duas informações:

- Mensagem → representação numérica → classificação conhecida

Por exemplo, imagine que durante o treinamento o modelo encontre várias mensagens como:

- "Congratulations! You won a prize"

- "Win cash now"

- "Free prize waiting for you"

O modelo começa a identificar que determinadas palavras aparecem com maior frequência em mensagens classificadas como spam.




### Naive Bayes

O **Naive Bayes** é um classificador probabilístico baseado no Teorema de Bayes.

O Teorema de Bayes nos permite calcular a probabilidade de uma hipótese
dada uma evidência:

$$
P(A \mid B)
=
\frac{P(B \mid A)P(A)}{P(B)}
$$

No problema de classificação de mensagens, queremos calcular a probabilidade
de uma mensagem pertencer a uma determinada classe a partir das palavras
que aparecem nela.

Quando temos várias características (palavras), o Naive Bayes assume,
de forma simplificada, que essas características são condicionalmente
independentes:

$$
P(C \mid x_1,x_2,\ldots,x_n)
=
\frac{
P(C)\prod_{i=1}^{n}P(x_i\mid C)
}{
P(x_1,x_2,\ldots,x_n)
}
$$

Onde:

- $C$ representa a classe da mensagem (`spam` ou `ham`);
- $x_1, x_2, \ldots, x_n$ representam as características da mensagem;
- $P(C)$ é a probabilidade inicial de uma determinada classe;
- $P(x_i \mid C)$ é a probabilidade de uma determinada característica
  aparecer naquela classe.

#### Por que "Naive"?

A hipótese "ingênua" está na suposição de que as características são
condicionalmente independentes.

Em textos, essa suposição não é necessariamente verdadeira: palavras
podem estar relacionadas entre si. Mesmo assim, essa simplificação
permite construir um classificador eficiente e, em muitos problemas de
classificação de texto, bastante útil.

#### Por que usar MultinomialNB?

No problema de classificação de mensagens de SPAM, as características são principalmente contagens de palavras.

Cada número indica quantas vezes determinada palavra aparece na mensagem.

O MultinomialNB é uma variante do Naive Bayes adequada para esse tipo de representação, em que trabalhamos com contagens ou frequências de características.


In [ ]:
modelo = MultinomialNB()

# X_train_vec: representação numérica das mensagens
# y_train: classificação correta dessas mensagens
modelo.fit(X_train_vec, y_train)

## Fazendo previsões

Depois de treinar o modelo, é importante verificar se ele consegue classificar mensagens que ele ainda não viu.

X_test_vec, que contém as mensagens do conjunto de teste transformadas em números pelo CountVectorizer será utilizada para realizar as previsões.



In [ ]:
previsoes = modelo.predict(X_test_vec)

## Avaliação do modelo

Depois que o modelo faz as previsões, é possível descobrir o quão boas foram essas previsões.

Temos duas informações:

- y_test: contém as respostas corretas do conjunto de teste
- previsoes: contém as respostas produzidas pelo modelo


### Acurácia (Accuracy)

Calcula a proporção de previsões corretas em relação ao total de mensagens.


A acurácia responde:

- De todas as mensagens testadas, em quantas o modelo acertou?


Por exemplo, se o modelo classificou corretamente 950 de 1.000 mensagens a acurácia será de 0,95 ou 95%.




In [ ]:
print("Acurácia:", accuracy_score(y_test, previsoes))

### Precisão (Precision)

Calcula a proporção de previsões corretas para a classe objetivo.

A precisão responde:
- Entre todas as mensagens que o modelo chamou de spam, quantas realmente eram spam?

Por exemplo, imagine que o modelo classificou 100 mensagens como spam, mas apenas 90 eram realmente spam. Nesse caso, a precisão será de 0,90.






In [ ]:
print("Precisão:", precision_score(y_test, previsoes))

### Revocação (Recall)


A revocação responde:
- Entre todas as mensagens que realmente eram spam, quantas o modelo conseguiu identificar?

Imagine que existam 100 mensagens realmente spam e o modelo consiga identificar 80 delas, então o recall será de 0,80.

In [ ]:
print("Recall:", recall_score(y_test, previsoes))

### F1-score
O F1-score combina precisão e recall em uma única métrica.

É uma métrica útil quando queremos considerar as duas coisas ao mesmo tempo:

- evitar classificar mensagens legítimas como spam;
- identificar o máximo possível de mensagens que realmente são spam.

O F1-score varia entre 0 e 1.



In [ ]:
print("F1:", f1_score(y_test, previsoes))

### Matriz de confusão

As métricas anteriores resumem o desempenho do modelo em alguns números. A matriz de confusão permite olhar com mais detalhe para quais tipos de erros o modelo cometeu.

No problema abordado, temos duas classes:
- 0: ham / não spam
- 1: spam

A matriz de confusão compara:
- classe real: o que a mensagem realmente é
- classe prevista: o que o modelo classificou




### Exercicio 6 - Análise de Matriz de Confusão

- Quantas mensagens foram classificadas corretamente como ham?

- Quantas mensagens foram classificadas corretamente como spam?

- Quantas mensagens ham foram classificadas incorretamente como spam?

- Quantas mensagens spam foram classificadas incorretamente como ham?

- Qual dos dois tipos de erro você considera mais preocupante neste problema específico? Justifique.





In [ ]:
matriz = confusion_matrix(y_test, previsoes)

sns.heatmap(
    matriz,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Não spam", "Spam"],
    yticklabels=["Não spam", "Spam"]
)

plt.xlabel("Classe prevista")
plt.ylabel("Classe real")
plt.title("Matriz de Confusão")
plt.show()

### Exercício 7 - Testando o modelo com novas mensagens

Execute o exemplo a seguir e, posteriormente, crie três mensagens próprias. Antes de executar o modelo, classifique cada uma como spam ou não spam. Depois compare sua classificação com a previsão do modelo.


In [ ]:
novas_mensagens = [
    "Congratulations! You won a free prize!",
    "Can you send me the homework?",
    "You have been selected for a cash reward!",
    "Are we meeting tomorrow after class?"
]

novas_mensagens_vec = vectorizer.transform(novas_mensagens)

novas_previsoes = modelo.predict(novas_mensagens_vec)

for mensagem, previsao in zip(novas_mensagens, novas_previsoes):
    categoria = "SPAM" if previsao == 1 else "NÃO SPAM"
    print(f"{categoria}: {mensagem}")
